**Step 1: Install Required Libraries**

In [1]:
import pandas as pd
import numpy as np
import re
import string
import time
import warnings
warnings.filterwarnings('ignore')

print("✅ All Libraries Imported.")

✅ All Libraries Imported.


**Step 2: Load and Explore Dataset**

In [2]:
df = pd.read_csv('Data Science Project 4.csv')

print("="*60)
print("DATASET EXPLORATION")
print("="*60)

print(f"\n📊 Total Reviews: {len(df)}")
print(f"📊 Columns: {df.columns.tolist()}")
print(f"📊 Data Types:\n{df.dtypes}")

print("\n📊 First 3 Reviews:")
print(df.head(3))

print("\n📊 Sentiment Distribution:")
print(df['sentiment'].value_counts())

print("\n✅ Dataset loaded.")

DATASET EXPLORATION

📊 Total Reviews: 50000
📊 Columns: ['review', 'sentiment']
📊 Data Types:
review       object
sentiment    object
dtype: object

📊 First 3 Reviews:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive

📊 Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

✅ Dataset loaded.


**Step 3: Text Cleaning Function**

In [3]:
import re
import string

def clean_text(text):
    """
    Clean and preprocess text data
    Steps:
    1. Convert to string
    2. Remove HTML tags
    3. Remove URLs
    4. Remove punctuation
    5. Convert to lowercase
    6. Remove extra spaces
    """
    # Convert to string
    text = str(text)

    # Remove HTML tags like <br />
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Convert to lowercase
    text = text.lower()

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply cleaning
df['cleaned_review'] = df['review'].apply(clean_text)

print("="*60)
print("TEXT CLEANING")
print("="*60)

print("\n📝 Original Review:")
print(df['review'][0][:200])

print("\n📝 Cleaned Review:")
print(df['cleaned_review'][0][:200])

print("\n✅ Text cleaned.")

TEXT CLEANING

📝 Original Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo

📝 Cleaned Review:
one of the other reviewers has mentioned that after watching just 1 oz episode youll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its b

✅ Text cleaned.


**Step 4: Encode Sentiment Labels**

In [4]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['sentiment_encoded'] = label_encoder.fit_transform(df['sentiment'])

print("="*60)
print("LABEL ENCODING")
print("="*60)

print("\n📊 Sentiment Mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"   {label} → {i}")

print("\n📊 Sample Encoded Data:")
print(df[['sentiment', 'sentiment_encoded']].head(10))

print("\n✅ Labels Encoded.")

LABEL ENCODING

📊 Sentiment Mapping:
   negative → 0
   positive → 1

📊 Sample Encoded Data:
  sentiment  sentiment_encoded
0  positive                  1
1  positive                  1
2  positive                  1
3  negative                  0
4  positive                  1
5  positive                  1
6  positive                  1
7  negative                  0
8  negative                  0
9  positive                  1

✅ Labels Encoded.


**Step 5: Split Data into Train/Test Sets**

In [5]:
from sklearn.model_selection import train_test_split

X = df['cleaned_review']  # Features
y = df['sentiment_encoded']  # Target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          # 20% for testing
    random_state=42,        # Reproducibility
    stratify=y              # Maintain class balance
)

print("="*60)
print("DATA SPLIT")
print("="*60)

print(f"\n📊 Training Set: {len(X_train)} Samples")
print(f"📊 Testing Set: {len(X_test)} Samples")

print(f"\n📊 Training Set Distribution:")
print(y_train.value_counts())

print(f"\n📊 Testing Set Distribution:")
print(y_test.value_counts())

print("\n✅ Data Split Done.")

DATA SPLIT

📊 Training Set: 40000 Samples
📊 Testing Set: 10000 Samples

📊 Training Set Distribution:
sentiment_encoded
1    20000
0    20000
Name: count, dtype: int64

📊 Testing Set Distribution:
sentiment_encoded
0    5000
1    5000
Name: count, dtype: int64

✅ Data Split Done.


**Step 6: TF-IDF Vectorization**

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
import time

print("="*60)
print("TF-IDF VECTORIZATION")
print("="*60)

print("\n⏳ Converting Text to Numbers...")
start_time = time.time()

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,      # Use top 5000 words
    min_df=2,              # Ignore words appearing in < 2 docs
    stop_words='english',  # Remove common words
    ngram_range=(1, 2)     # Use single words and 2-word combinations
)

# Fit on training data and transform
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data using same vocabulary
X_test_tfidf = tfidf.transform(X_test)

elapsed = time.time() - start_time

print(f"\n✅ TF-IDF Matrix Shape (Training): {X_train_tfidf.shape}")
print(f"✅ TF-IDF Matrix Shape (Testing): {X_test_tfidf.shape}")
print(f"✅ Time taken: {elapsed:.2f} Seconds")

# Show sample features
feature_names = tfidf.get_feature_names_out()
print(f"\n📊 Sample Features (First 10):")
print(feature_names[:10])

print("\n✅ Text Converted to Numbers.")

TF-IDF VECTORIZATION

⏳ Converting Text to Numbers...

✅ TF-IDF Matrix Shape (Training): (40000, 5000)
✅ TF-IDF Matrix Shape (Testing): (10000, 5000)
✅ Time taken: 30.32 Seconds

📊 Sample Features (First 10):
['10' '10 10' '10 minutes' '10 years' '100' '1010' '11' '110' '12' '13']

✅ Text Converted to Numbers.


**Step 7: Train Naive Bayes Model**

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("="*60)
print("NAIVE BAYES MODEL")
print("="*60)

print("\n⏳ Training Naive Bayes...")
start_time = time.time()

nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)

elapsed = time.time() - start_time

print(f"\n✅ Naive Bayes Trained in {elapsed:.2f} Seconds")

# Make predictions
nb_predictions = nb_model.predict(X_test_tfidf)

# Evaluate
nb_accuracy = accuracy_score(y_test, nb_predictions)

print(f"\n📊 Naive Bayes Accuracy: {nb_accuracy:.4f} ({nb_accuracy*100:.2f}%)")

print("\n📊 Classification Report:")
print(classification_report(y_test, nb_predictions, target_names=label_encoder.classes_))

print("\n📊 Confusion Matrix:")
cm_nb = confusion_matrix(y_test, nb_predictions)
print(pd.DataFrame(cm_nb, index=label_encoder.classes_, columns=label_encoder.classes_))

print("\n✅ Naive Bayes Trained.")

NAIVE BAYES MODEL

⏳ Training Naive Bayes...

✅ Naive Bayes Trained in 0.04 Seconds

📊 Naive Bayes Accuracy: 0.8611 (86.11%)

📊 Classification Report:
              precision    recall  f1-score   support

    negative       0.87      0.84      0.86      5000
    positive       0.85      0.88      0.86      5000

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000


📊 Confusion Matrix:
          negative  positive
negative      4219       781
positive       608      4392

✅ Naive Bayes Trained.


**Step 8: Train SVM Model**

In [8]:
from sklearn.svm import LinearSVC

print("="*60)
print("SVM MODEL")
print("="*60)

print("\n⏳ Training SVM...")
start_time = time.time()

svm_model = LinearSVC(
    C=1.0,
    max_iter=1000,
    dual=False,
    random_state=42,
    tol=1e-4
)

svm_model.fit(X_train_tfidf, y_train)

elapsed = time.time() - start_time

print(f"\n✅ SVM trained in {elapsed:.2f} Seconds")

# Make predictions
svm_predictions = svm_model.predict(X_test_tfidf)

# Evaluate
svm_accuracy = accuracy_score(y_test, svm_predictions)

print(f"\n📊 SVM Accuracy: {svm_accuracy:.4f} ({svm_accuracy*100:.2f}%)")

print("\n📊 Classification Report:")
print(classification_report(y_test, svm_predictions, target_names=label_encoder.classes_))

print("\n📊 Confusion Matrix:")
cm_svm = confusion_matrix(y_test, svm_predictions)
print(pd.DataFrame(cm_svm, index=label_encoder.classes_, columns=label_encoder.classes_))

print("\n✅ SVM trained.")

SVM MODEL

⏳ Training SVM...

✅ SVM trained in 1.21 Seconds

📊 SVM Accuracy: 0.8835 (88.35%)

📊 Classification Report:
              precision    recall  f1-score   support

    negative       0.89      0.88      0.88      5000
    positive       0.88      0.89      0.88      5000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


📊 Confusion Matrix:
          negative  positive
negative      4382       618
positive       547      4453

✅ SVM trained.


**Step 9: Train Logistic Regression Model**

In [9]:
from sklearn.linear_model import LogisticRegression

print("="*60)
print("LOGISTIC REGRESSION MODEL")
print("="*60)

print("\n⏳ Training Logistic Regression...")
start_time = time.time()

lr_model = LogisticRegression(
    max_iter=500,
    random_state=42,
    C=1.0,
    solver='liblinear'
)

lr_model.fit(X_train_tfidf, y_train)

elapsed = time.time() - start_time

print(f"\n✅ Logistic Regression Trained in {elapsed:.2f} Seconds")

# Make predictions
lr_predictions = lr_model.predict(X_test_tfidf)

# Evaluate
lr_accuracy = accuracy_score(y_test, lr_predictions)

print(f"\n📊 Logistic Regression Accuracy: {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")

print("\n📊 Classification Report:")
print(classification_report(y_test, lr_predictions, target_names=label_encoder.classes_))

print("\n✅ Logistic Regression Trained.")

LOGISTIC REGRESSION MODEL

⏳ Training Logistic Regression...

✅ Logistic Regression Trained in 1.54 Seconds

📊 Logistic Regression Accuracy: 0.8936 (89.36%)

📊 Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      5000
    positive       0.89      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


✅ Logistic Regression Trained.


**Step 10: Compare All Models**

In [10]:
print("="*60)
print("MODEL COMPARISON")
print("="*60)

# Create comparison table
comparison = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM', 'Logistic Regression'],
    'Accuracy': [nb_accuracy, svm_accuracy, lr_accuracy],
    'Training Time': ['Fast', 'Medium', 'Fast']
})

print("\n📊 Model Performance Comparison:")
print(comparison.to_string(index=False))

# Find best model
best_idx = comparison['Accuracy'].idxmax()
best_model_name = comparison.loc[best_idx, 'Model']
best_accuracy = comparison.loc[best_idx, 'Accuracy']

print(f"\n🏆 Best Model: {best_model_name}")
print(f"🏆 Best Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")

# Select best model
if best_model_name == 'Naive Bayes':
    best_model = nb_model
elif best_model_name == 'SVM':
    best_model = svm_model
else:
    best_model = lr_model

print("\n✅ Model Comparison Done.")

MODEL COMPARISON

📊 Model Performance Comparison:
              Model  Accuracy Training Time
        Naive Bayes    0.8611          Fast
                SVM    0.8835        Medium
Logistic Regression    0.8936          Fast

🏆 Best Model: Logistic Regression
🏆 Best Accuracy: 0.8936 (89.36%)

✅ Model Comparison Done.


**Step 11: Find Most Important Features**

In [11]:

print("="*60)
print("FEATURE IMPORTANCE")
print("="*60)

# Get feature names
feature_names = tfidf.get_feature_names_out()

# For SVM or Logistic Regression, get coefficients
if best_model_name == 'SVM' or best_model_name == 'Logistic Regression':
    coef = best_model.coef_[0]

    # Top words for POSITIVE sentiment
    top_positive_idx = coef.argsort()[-10:][::-1]

    print("\n🔝 Top 10 Words for POSITIVE Sentiment:")
    for idx in top_positive_idx:
        if feature_names[idx]:
            print(f"   • {feature_names[idx]}: {coef[idx]:.4f}")

    # Top words for NEGATIVE sentiment
    top_negative_idx = coef.argsort()[:10]

    print("\n🔝 Top 10 Words for NEGATIVE Sentiment:")
    for idx in top_negative_idx:
        if feature_names[idx]:
            print(f"   • {feature_names[idx]}: {coef[idx]:.4f}")

print("\n✅ Feature Importance Shown.")

FEATURE IMPORTANCE

🔝 Top 10 Words for POSITIVE Sentiment:
   • excellent: 6.8118
   • great: 6.6071
   • 710: 6.1491
   • perfect: 5.1070
   • amazing: 5.0967
   • best: 5.0392
   • wonderful: 4.6729
   • hilarious: 4.4185
   • favorite: 4.2799
   • loved: 4.2493

🔝 Top 10 Words for NEGATIVE Sentiment:
   • worst: -9.4488
   • awful: -7.5449
   • waste: -6.9680
   • bad: -6.9379
   • boring: -6.3287
   • poor: -5.5181
   • terrible: -5.4213
   • dull: -5.2288
   • worse: -5.1484
   • poorly: -5.1282

✅ Feature Importance Shown.


**Step 12: Create Prediction Function**

In [12]:

def predict_sentiment(text):
    """
    Predict sentiment of a new review

    Parameters:
    text: str, the review text

    Returns:
    sentiment: str ('positive' or 'negative')
    probability: float, confidence level
    """
    # Step 1: Clean the text
    cleaned = clean_text(text)

    # Step 2: Convert to TF-IDF
    features = tfidf.transform([cleaned])

    # Step 3: Make prediction using best model
    prediction = best_model.predict(features)[0]

    # Step 4: Get probability if available
    try:
        probabilities = best_model.predict_proba(features)[0]
        prob = max(probabilities)
    except:
        prob = None

    # Step 5: Convert to label
    sentiment = label_encoder.inverse_transform([prediction])[0]

    return sentiment, prob

print("="*60)
print("PREDICTION FUNCTION")
print("="*60)

print("\n✅ Prediction Function Created!")
print("✅ Function: predict_sentiment(text)")
print("✅ Returns: (sentiment, confidence)")

PREDICTION FUNCTION

✅ Prediction Function Created!
✅ Function: predict_sentiment(text)
✅ Returns: (sentiment, confidence)


**Step 13: Test on New Reviews**

In [13]:

print("="*60)
print("TESTING PREDICTIONS")
print("="*60)

test_reviews = [
    "This movie was absolutely fantastic! I loved every minute of it. The acting was superb and the story was gripping.",
    "Terrible film, complete waste of time. Boring and predictable plot with wooden acting.",
    "The acting was okay, but the story could have been much better.",
    "Amazing cinematography and great performances by all actors. A must watch!",
    "I didn't like this movie at all. Very disappointing and overrated.",
    "An excellent movie with brilliant performances. Highly recommended to everyone!",
    "Average film, nothing special. Some good parts but mostly forgettable."
]

print("\n📝 Testing 7Rreviews:\n")

for i, review in enumerate(test_reviews, 1):
    sentiment, confidence = predict_sentiment(review)
    print(f"{i}. Review: {review[:60]}...")
    print(f"   → Sentiment: {sentiment.upper()}")
    if confidence:
        print(f"   → Confidence: {confidence:.2%}")
    print()

TESTING PREDICTIONS

📝 Testing 7Rreviews:

1. Review: This movie was absolutely fantastic! I loved every minute of...
   → Sentiment: POSITIVE
   → Confidence: 96.71%

2. Review: Terrible film, complete waste of time. Boring and predictabl...
   → Sentiment: NEGATIVE
   → Confidence: 100.00%

3. Review: The acting was okay, but the story could have been much bett...
   → Sentiment: NEGATIVE
   → Confidence: 94.05%

4. Review: Amazing cinematography and great performances by all actors....
   → Sentiment: POSITIVE
   → Confidence: 98.95%

5. Review: I didn't like this movie at all. Very disappointing and over...
   → Sentiment: NEGATIVE
   → Confidence: 94.19%

6. Review: An excellent movie with brilliant performances. Highly recom...
   → Sentiment: POSITIVE
   → Confidence: 99.92%

7. Review: Average film, nothing special. Some good parts but mostly fo...
   → Sentiment: NEGATIVE
   → Confidence: 84.43%



**Step 14: Save Model for Future Use**

In [14]:
import joblib
import os

print("="*60)
print("STEP 14: SAVING MODEL")
print("="*60)

# Create models directory if it doesn't exist
if not os.path.exists('models'):
    os.makedirs('models')

# Save best model
joblib.dump(best_model, 'models/best_sentiment_model.pkl')

# Save TF-IDF vectorizer
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')

# Save label encoder
joblib.dump(label_encoder, 'models/label_encoder.pkl')

print("\n✅ Model Saved Successfully!")
print("\n📁 Files Created:")
print("   • models/best_sentiment_model.pkl")
print("   • models/tfidf_vectorizer.pkl")
print("   • models/label_encoder.pkl")

# Verify files
print("\n📊 File s\Sizes:")
for file in ['best_sentiment_model.pkl', 'tfidf_vectorizer.pkl', 'label_encoder.pkl']:
    path = f'models/{file}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"   • {file}: {size:.1f} KB")

print("\n✅ Model saved.")

STEP 14: SAVING MODEL

✅ Model Saved Successfully!

📁 Files Created:
   • models/best_sentiment_model.pkl
   • models/tfidf_vectorizer.pkl
   • models/label_encoder.pkl

📊 File s\Sizes:
   • best_sentiment_model.pkl: 39.9 KB
   • tfidf_vectorizer.pkl: 181.4 KB
   • label_encoder.pkl: 0.5 KB

✅ Model saved.


**Step 15: Load and Use Saved Model**

In [17]:
print("="*60)
print("LOADING SAVED MODEL")
print("="*60)

print("""
📌 Code to load saved model:

```python
import joblib

# Load model and components
loaded_model = joblib.load('models/best_sentiment_model.pkl')
loaded_vectorizer = joblib.load('models/tfidf_vectorizer.pkl')
loaded_encoder = joblib.load('models/label_encoder.pkl')

# Prediction function
def predict_saved(text):
    cleaned = clean_text(text)
    features = loaded_vectorizer.transform([cleaned])
    pred = loaded_model.predict(features)[0]
    return loaded_encoder.inverse_transform([pred])[0]

# Test
review = "This movie is amazing!"
sentiment = predict_saved(review)
print(f"Sentiment: {sentiment}")
""")

print("\n✅ Loading Demonstrated.")

df.to_csv('Cleaned Data Science Project 4.csv', index=False)

LOADING SAVED MODEL

📌 Code to load saved model:

```python
import joblib

# Load model and components
loaded_model = joblib.load('models/best_sentiment_model.pkl')
loaded_vectorizer = joblib.load('models/tfidf_vectorizer.pkl')
loaded_encoder = joblib.load('models/label_encoder.pkl')

# Prediction function
def predict_saved(text):
    cleaned = clean_text(text)
    features = loaded_vectorizer.transform([cleaned])
    pred = loaded_model.predict(features)[0]
    return loaded_encoder.inverse_transform([pred])[0]

# Test
review = "This movie is amazing!"
sentiment = predict_saved(review)
print(f"Sentiment: {sentiment}")


✅ Loading Demonstrated.
